# TernakKu — PoC AI Risiko Kesehatan & Inbreeding Alert
**Kelompok 9** · LKP 6 Section 2 (AI Feasibility Check)

Notebook ini memakai dataset publik Kaggle **Cattle Health and Feeding Data** (shahhet2812) sebagai **data proxy**.
Dataset ini **sintetis** (tertulis di halaman Kaggle-nya) dan lisensinya *Unknown*, jadi angka di sini adalah
gambaran awal pipeline, **bukan bukti performa di lapangan**.

Isi notebook:
1. Muat data dan bentuk label biner `sick`
2. Feature engineering (selisih produksi susu, jumlah vaksin lengkap)
3. Split per `Farm_ID` (60/20/20) supaya tidak ada kebocoran antar peternakan
4. Baseline (kelas mayoritas dan aturan sederhana) vs Logistic Regression, Random Forest, Gradient Boosting
5. Pemilihan threshold untuk target recall dan level Rendah / Sedang / Tinggi
6. Explainability (faktor utama) dan contoh output sesuai kontrak API
7. Ablation: apakah tanda vital membantu?
8. Inbreeding Alert (rule-based) dengan contoh silsilah
9. Simpan model

## 1. Setup dan muat data

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

CSV = "global_cattle_disease_detection_dataset.csv"
SAMPLE_N = None        # isi mis. 60000 kalau Colab terasa lambat; None = pakai semua 250.000 baris
RANDOM_STATE = 42
TARGET_RECALL = 0.85   # target SRS: recall kelas Tinggi >= 85%
TARGET_PRECISION = 0.75

try:
    from google.colab import files
    if not Path(CSV).exists():
        print("Upload file", CSV)
        files.upload()
except ImportError:
    pass  # bukan Colab, pakai file lokal

df = pd.read_csv(CSV)
if SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
print(df.shape, "| missing:", int(df.isna().sum().sum()), "| duplikat ID:", int(df.Cattle_ID.duplicated().sum()))
df.head(3)

## 2. Label dan feature engineering
- Label `sick` = 1 jika `Disease_Status` bukan *Healthy*. Ke-44 penyakit digabung karena masing-masing hanya ~1% data.
- `yield_drop_ratio` = penurunan produksi susu dibanding rata-rata minggu lalu.
- `vaccine_count` = jumlah vaksin yang sudah lengkap dari 8 jenis (data ini hanya punya status 0/1, tanpa tanggal).

In [ ]:
vacc_cols = [c for c in df.columns if c.endswith("_Vaccine")]
df["sick"] = (df["Disease_Status"] != "Healthy").astype(int)
df["vaccine_count"] = df[vacc_cols].sum(axis=1)
df["yield_drop_ratio"] = ((df["Previous_Week_Avg_Yield"] - df["Milk_Yield_L"])
                          / df["Previous_Week_Avg_Yield"].clip(lower=0.1)).clip(-1, 1)

NUM = ["Age_Months", "Parity", "Days_in_Milk", "Body_Condition_Score", "Milk_Yield_L",
       "Previous_Week_Avg_Yield", "yield_drop_ratio", "vaccine_count",
       "Feed_Quantity_kg", "Water_Intake_L", "Housing_Score"]
CAT = ["Lactation_Stage", "Season", "Management_System"]
VITAL = ["Body_Temperature_C", "Heart_Rate_bpm", "Respiratory_Rate", "Rumination_Time_hrs"]

print("Proporsi sakit:", round(df.sick.mean(), 3))
print(df.groupby("sick")[["Milk_Yield_L", "Previous_Week_Avg_Yield", "vaccine_count", "Body_Condition_Score"]].mean().round(2))

## 3. Split per peternakan (Farm_ID)
Satu peternakan hanya masuk satu bagian (train / validasi / test) supaya model diuji pada peternakan yang belum pernah dilihat.
Validasi dipakai untuk memilih threshold, test hanya dipakai sekali di akhir.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
tv_idx, te_idx = next(gss.split(df, groups=df["Farm_ID"]))
tv, te = df.iloc[tv_idx], df.iloc[te_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)  # 0.25 x 0.8 = 20%
tr_idx, va_idx = next(gss2.split(tv, groups=tv["Farm_ID"]))
tr, va = tv.iloc[tr_idx], tv.iloc[va_idx]

for n, d in [("train", tr), ("val", va), ("test", te)]:
    print(f"{n:5s} n={len(d):6d} sakit={d.sick.mean():.3f} farm={d.Farm_ID.nunique()}")
assert not set(tr.Farm_ID) & set(te.Farm_ID) and not set(va.Farm_ID) & set(te.Farm_ID)

## 4. Baseline dan model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)

def make_pre(num, cat, onehot=True):
    enc = OneHotEncoder(handle_unknown="ignore") if onehot else OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    return ColumnTransformer([("num", StandardScaler() if onehot else "passthrough", num), ("cat", enc, cat)])

def evaluate(name, y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return dict(model=name, thr=round(float(thr), 3), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred), f1=f1_score(y, pred),
                roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p))

FEATS = NUM + CAT
Xtr, ytr, Xva, yva, Xte, yte = tr[FEATS], tr.sick, va[FEATS], va.sick, te[FEATS], te.sick

models = {
    "LogReg": Pipeline([("pre", make_pre(NUM, CAT)), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "RandomForest": Pipeline([("pre", make_pre(NUM, CAT, onehot=False)),
                              ("clf", RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1,
                                                             class_weight="balanced_subsample", random_state=RANDOM_STATE))]),
    "HistGB": Pipeline([("pre", make_pre(NUM, CAT, onehot=False)),
                        ("clf", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, random_state=RANDOM_STATE))]),
}
for m in models.values():
    m.fit(Xtr, ytr)

rows = []
# Baseline 1: selalu prediksi kelas mayoritas (Sehat)
rows.append(dict(model="Baseline: mayoritas", thr=np.nan, precision=0.0, recall=0.0, f1=0.0, roc_auc=0.5, pr_auc=float(yva.mean())))
# Baseline 2: aturan sederhana (susu turun > 20% dari minggu lalu)
rule = (va["yield_drop_ratio"] > 0.20).astype(int)
rows.append(dict(model="Baseline: aturan susu turun>20%", thr=np.nan, precision=precision_score(yva, rule), recall=recall_score(yva, rule),
                 f1=f1_score(yva, rule), roc_auc=roc_auc_score(yva, va["yield_drop_ratio"]), pr_auc=average_precision_score(yva, va["yield_drop_ratio"])))
val_proba = {}
for name, m in models.items():
    val_proba[name] = m.predict_proba(Xva)[:, 1]
    rows.append(evaluate(name, yva, val_proba[name], thr=0.5))
res_val = pd.DataFrame(rows).set_index("model").round(3)
print("HASIL PADA DATA VALIDASI (threshold 0.5)")
res_val

## 5. Threshold dan level risiko (Rendah / Sedang / Tinggi)
Ada dua target di SRS yang **saling bertabrakan**: recall >= 85% dan precision >= 75%. Di data ini keduanya tidak bisa dicapai bersamaan
(lihat kurva di bawah), jadi level risiko dibuat dengan dua batas, keduanya dipilih di data **validasi**:
- **Batas Sedang** = threshold terendah dengan recall >= 85% untuk gabungan Sedang+Tinggi (jangan sampai ternak sakit lolos jadi Rendah).
- **Batas Tinggi** = threshold dengan precision >= `TIER_PRECISION` (alarm Tinggi harus cukup bisa dipercaya).

Model yang dipakai di aplikasi adalah **Logistic Regression** karena penjelasan faktornya bisa dihitung dari koefisien.
Hasilnya lalu diukur sekali di data **test**.

In [ ]:
TIER_PRECISION = 0.65   # precision minimum untuk level Tinggi (dipilih dari kurva PR, bukan dari SRS)
DEPLOY = "LogReg"
best_name = res_val.loc[list(models), "pr_auc"].idxmax()
print("Model terbaik di validasi (PR-AUC):", best_name, "| model yang dipakai di aplikasi:", DEPLOY)

def pick_threshold(y, p, target_recall):
    prec, rec, thr = precision_recall_curve(y, p)
    ok = np.where(rec[:-1] >= target_recall)[0]
    return thr[ok[-1]] if len(ok) else thr[0]

def pick_precision_threshold(y, p, target_precision):
    prec, rec, thr = precision_recall_curve(y, p)
    ok = np.where(prec[:-1] >= target_precision)[0]
    return thr[ok[0]] if len(ok) else thr[-1]

rows = []
for name, m in models.items():
    t = pick_threshold(yva, val_proba[name], TARGET_RECALL)
    p_t = m.predict_proba(Xte)[:, 1]
    rows.append(evaluate(name + " (thr utk recall>=85%)", yte, p_t, thr=t))
    rows.append(evaluate(name + " (thr 0.5)", yte, p_t, thr=0.5))
res_te = pd.DataFrame(rows).set_index("model").round(3)
print("HASIL PADA DATA TEST")
display(res_te)

dep = models[DEPLOY]
p_va = val_proba[DEPLOY]
p_te = dep.predict_proba(Xte)[:, 1]
t_mid = float(pick_threshold(yva, p_va, TARGET_RECALL))              # batas Sedang
t_high = float(pick_precision_threshold(yva, p_va, TIER_PRECISION))  # batas Tinggi
print(f"Batas Sedang = {t_mid:.3f} | Batas Tinggi = {t_high:.3f}")

In [ ]:
def risk_level(p):
    return np.where(p >= t_high, "Tinggi", np.where(p >= t_mid, "Sedang", "Rendah"))

tier = pd.DataFrame({"level": risk_level(p_te), "sick": yte.values})
summary = tier.groupby("level").sick.agg(n="size", persen_sakit="mean").reindex(["Rendah", "Sedang", "Tinggi"])
summary["porsi_ternak"] = (summary.n / len(tier)).round(3)
summary["persen_sakit"] = summary.persen_sakit.round(3)
print("Persentase ternak benar-benar sakit per level (data test). Kalau model berguna, Rendah < Sedang < Tinggi.")
display(summary)

flag_s = (p_te >= t_mid).astype(int)   # Sedang atau Tinggi
flag_t = (p_te >= t_high).astype(int)  # hanya Tinggi
print(f"Sedang+Tinggi: recall {recall_score(yte, flag_s):.3f} | precision {precision_score(yte, flag_s):.3f}")
print(f"Hanya Tinggi : recall {recall_score(yte, flag_t):.3f} | precision {precision_score(yte, flag_t):.3f}  (SRS: recall>=0.85, precision>=0.75)")
print("Confusion matrix Tinggi-saja (baris=aktual sehat/sakit, kolom=prediksi):")
print(confusion_matrix(yte, flag_t))
ok = recall_score(yte, flag_t) >= TARGET_RECALL and precision_score(yte, flag_t) >= TARGET_PRECISION
print("-> SESUAI TARGET SRS" if ok else "-> DI BAWAH TARGET SRS (tulis apa adanya di laporan)")

In [ ]:
prec, rec, thr = precision_recall_curve(yva, p_va)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rec, prec); ax[0].axvline(TARGET_RECALL, ls="--", c="r", lw=1); ax[0].axhline(TARGET_PRECISION, ls="--", c="g", lw=1)
ax[0].axhline(yva.mean(), ls=":", c="gray"); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision")
ax[0].set_title(f"Kurva PR ({DEPLOY}, validasi)\nmerah=target recall, hijau=target precision, abu=acak")
ax[1].hist(p_te[yte == 0], bins=40, alpha=.6, label="sehat"); ax[1].hist(p_te[yte == 1], bins=40, alpha=.6, label="sakit")
ax[1].axvline(t_mid, c="orange", ls="--"); ax[1].axvline(t_high, c="r", ls="--")
ax[1].set_title("Distribusi probabilitas sakit (test)\noranye=batas Sedang, merah=batas Tinggi"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Explainability dan contoh output API
Logistic Regression dipakai untuk penjelasan karena kontribusi tiap fitur bisa dihitung dari koefisien × nilai (setelah di-scale).
Output mengikuti kontrak API `POST /api/v1/risk-score` di LKP 6 (`risk_level`, `confidence`, `top_factors`, `recommended_actions`, `is_estimate`).

In [ ]:
lr = models["LogReg"]
pre, clf = lr.named_steps["pre"], lr.named_steps["clf"]
names = pre.get_feature_names_out()

LABEL = {"Milk_Yield_L": "produksi susu rendah", "Previous_Week_Avg_Yield": "rata-rata susu minggu lalu rendah",
         "yield_drop_ratio": "produksi susu turun dari minggu lalu", "vaccine_count": "vaksin belum lengkap",
         "Body_Condition_Score": "kondisi tubuh (BCS) kurang baik", "Age_Months": "umur ternak",
         "Parity": "jumlah kelahiran (paritas)", "Days_in_Milk": "hari laktasi",
         "Feed_Quantity_kg": "jumlah pakan", "Water_Intake_L": "konsumsi air", "Housing_Score": "kualitas kandang"}
ACTION = {"Tinggi": "Periksa kondisi ternak hari ini dan hubungi penyuluh atau dokter hewan.",
          "Sedang": "Pantau ternak 2-3 hari ke depan dan lengkapi vaksinasi yang tertinggal.",
          "Rendah": "Lanjutkan pencatatan rutin."}

def risk_score(row: pd.DataFrame, livestock_id="SPI-012"):
    x = pre.transform(row[FEATS])
    p = float(lr.predict_proba(row[FEATS])[0, 1])
    contrib = np.asarray(x.todense() if hasattr(x, "todense") else x).ravel() * clf.coef_[0]
    order = np.argsort(-contrib)[:3]
    pos = [(names[i], contrib[i]) for i in order if contrib[i] > 0] or [(names[order[0]], abs(contrib[order[0]]))]
    tot = sum(c for _, c in pos)
    factors = []
    for n, c in pos:
        key = n.split("__", 1)[1]
        factors.append({"faktor": LABEL.get(key, key.replace("_", " ")), "kontribusi": round(float(c / tot), 2)})
    level = "Tinggi" if p >= t_high else ("Sedang" if p >= t_mid else "Rendah")
    return {"livestock_id": livestock_id, "risk_level": level, "confidence": round(max(p, 1 - p), 2),
            "prob_sakit": round(p, 3), "top_factors": factors, "recommended_actions": ACTION[level], "is_estimate": True}

# contoh: ambil satu ternak dari data test yang diprediksi Tinggi
ex = te[(p_te >= t_high)].head(1)
print(json.dumps(risk_score(ex, "SPI-012"), indent=2, ensure_ascii=False))

## 7. Ablation: apakah tanda vital membantu?
Secara klinis suhu dan detak jantung penting, tetapi dataset ini sintetis. Cek apakah ada sinyal di data ini.
Kalau AUC tidak naik, tulis di laporan bahwa tanda vital tidak berkontribusi **di dataset ini** (bukan berarti tidak penting di lapangan).

In [ ]:
rows = []
for label, feats_num in [("tanpa tanda vital", NUM), ("dengan tanda vital", NUM + VITAL)]:
    m = Pipeline([("pre", make_pre(feats_num, CAT)), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    m.fit(tr[feats_num + CAT], ytr)
    p = m.predict_proba(va[feats_num + CAT])[:, 1]
    rows.append(dict(setup=label, roc_auc=roc_auc_score(yva, p), pr_auc=average_precision_score(yva, p)))
pd.DataFrame(rows).set_index("setup").round(4)

## 8. Inbreeding Alert (rule-based)
Dataset Kaggle ini tidak punya silsilah, jadi bagian ini diuji dengan silsilah contoh. Untuk data yang lebih besar bisa dipakai
`pedigree.txt` dari dataset *Synthetic genomic data* (oubush) dengan format `anak, ayah, ibu`.
Koefisien kekerabatan memakai pendekatan Wright: Σ 0.5^(a+b+1) untuk tiap leluhur bersama (a, b = jumlah generasi dari kedua tetua).

In [ ]:
def ancestors(animal, ped, maxgen=4):
    """{leluhur: jumlah generasi di atas 'animal'}"""
    out, frontier = {}, [(animal, 0)]
    while frontier:
        x, g = frontier.pop(0)
        if g >= maxgen:
            continue
        for parent in ped.get(x, (None, None)):
            if parent and (parent not in out or g + 1 < out[parent]):
                out[parent] = g + 1
                frontier.append((parent, g + 1))
    return out

def inbreeding_check(sire_id, dam_id, ped, maxgen=4):
    A = {sire_id: 0, **ancestors(sire_id, ped, maxgen)}
    B = {dam_id: 0, **ancestors(dam_id, ped, maxgen)}
    common = set(A) & set(B)
    if not common:
        return {"is_related": False, "relation": "tidak sekerabat (dalam %d generasi)" % maxgen,
                "common_ancestor_generations": None, "kinship": 0.0, "message": "Tidak ditemukan hubungan kekerabatan dekat."}
    # hitung hanya leluhur bersama terdekat; leluhur yang hanya lewat leluhur bersama lain tidak dihitung dobel
    def anc(c): return set(ancestors(c, ped, maxgen))
    direct = [c for c in common if not any(c in anc(o) for o in common if o != c)]
    kin = sum(0.5 ** (A[c] + B[c] + 1) for c in direct)
    nearest = min(common, key=lambda c: A[c] + B[c])
    a, b = A[nearest], B[nearest]
    n_parents_common = sum(1 for c in direct if A[c] == 1 and B[c] == 1)
    if (a, b) in [(0, 1), (1, 0)]:
        rel = "orang tua dan anak"
    elif n_parents_common >= 2:
        rel = "saudara kandung"
    elif (a, b) == (1, 1):
        rel = "saudara tiri"
    else:
        rel = "kerabat (leluhur bersama %d dan %d generasi)" % (a, b)
    return {"is_related": True, "relation": rel, "common_ancestor_generations": max(a, b),
            "kinship": round(kin, 4), "message": "Pejantan dan induk masih sekerabat."}

# pedigree contoh: anak -> (ayah, ibu)
ped = {"SPI-003": ("SPI-001", "SPI-002"), "SPI-012": ("SPI-001", "SPI-005"),
       "SPI-020": ("SPI-030", "SPI-031"), "SPI-040": ("SPI-003", "SPI-020"), "SPI-041": ("SPI-003", "SPI-020"),
       "SPI-050": ("SPI-040", "SPI-060")}
tests = [("SPI-003", "SPI-012"), ("SPI-040", "SPI-041"), ("SPI-001", "SPI-003"), ("SPI-050", "SPI-012"), ("SPI-020", "SPI-012")]
for s, d in tests:
    print(s, "x", d, "->", json.dumps(inbreeding_check(s, d, ped), ensure_ascii=False))

## 9. Simpan model untuk FastAPI

In [ ]:
joblib.dump(lr, "risk_model_logreg.joblib")
meta = {"features_num": NUM, "features_cat": CAT, "threshold_high_logreg": float(t_high), "threshold_mid": float(t_mid),
        "best_model": best_name, "note": "Dilatih pada data sintetis Kaggle (proxy). Bukan bukti performa lapangan."}
Path("risk_model_meta.json").write_text(json.dumps(meta, indent=2))
print("Tersimpan:", [p.name for p in Path(".").glob("risk_model*")])
try:
    from google.colab import files
    for f in ["risk_model_logreg.joblib", "risk_model_meta.json"]:
        files.download(f)
except ImportError:
    pass

## Catatan untuk laporan (Bab 3, 4, 7)
- Tulis jujur bahwa data **sintetis, lisensi Unknown**; jumlah data nyata = jumlah baris yang dipakai (lihat output cell 1).
- Bandingkan model dengan **baseline** (tabel cell 4) untuk Bab 4.3.
- Target recall 85% / precision 75% (SRS) dibandingkan dengan hasil test (cell 5) untuk Bab 7.4. Kalau di bawah target, tulis penyesuaian (label lebih baik, data asli, ganti model, atau hanya badge Rendah/Tinggi).
- Hasil ablation (cell 7) menjelaskan mengapa tanda vital tidak dipakai di model ini.